# Sequential과 Router로 이해하는 Multi-Agent Collaboration

Multi-Agent는 역할이 다른 Agent를 연결해 하나의 요청을 처리하는 구조이다. 이 실습에서는 같은 Researcher와 ChartGenerator를 사용하되, 두 Worker를 항상 순서대로 실행하는 **Sequential**과 요청에 맞는 한 Worker만 고르는 **Router**를 비교한다.

## 빈칸 실습 안내

패키지 설치와 환경 설정 코드는 제공한다. 나머지 코드셀은 각 단계의 입력, 변환, 출력과 다음 사용처를 읽고 위에서 아래로 직접 완성한다.


## 패키지 설치

LangGraph와 LangChain OpenAI 연동, Tavily 검색, 차트 출력에 필요한 패키지를 준비한다.

- 사용할 패키지: `langgraph`, `langchain`, `langchain-openai`, `langchain-tavily`, `python-dotenv`, `matplotlib`
- 입력: 설치할 패키지 이름이다.
- 출력: 이후 셀에서 import할 수 있는 실행 환경이다.
- 다음 사용처: 환경 설정, Worker와 Tool 정의 단계이다.

In [1]:
%pip install -U langchain langchain-openai langgraph langchain-tavily matplotlib python-dotenv


   ---------------------------------------- 0.0/570.0 kB ? eta -:--:--
   ---------------------------------------- 570.0/570.0 kB 9.9 MB/s  0:00:00

  Attempting uninstall: langchain-core

    Found existing installation: langchain-core 1.5.6

   ---------------------------------------- 0/2 [langchain-core]
    Uninstalling langchain-core-1.5.6:
   ---------------------------------------- 0/2 [langchain-core]
      Successfully uninstalled langchain-core-1.5.6
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ------------


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Worker가 사용할 환경 변수와 모델 ID

현재 작업 디렉터리에서 `.env`를 찾아 환경 변수를 불러오고, 선택적으로 LangSmith 추적을 켠다. 모델 이름은 뒤에서 두 Worker가 공유한다.

- 관련 API: `find_dotenv`, `load_dotenv`, `os.getenv`, `os.environ.setdefault`
- 입력: `.env` 위치와 이미 등록된 환경 변수이다.
- 출력: `OPENAI_CHAT_MODEL`과 선택적인 추적 설정이다.
- 다음 사용처: `ChatOpenAI` 모델 생성 단계이다.

비밀값은 출력하지 않는다.

In [2]:
import os
from dotenv import find_dotenv, load_dotenv

dotenv_path = find_dotenv(usecwd=True)
if dotenv_path:
    load_dotenv(dotenv_path, override=False)

if os.getenv("LANGSMITH_API_KEY"):
    os.environ.setdefault("LANGSMITH_TRACING", "true")
    os.environ.setdefault("LANGSMITH_PROJECT", f"langgraph-collaboration")

OPENAI_CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna")


## Worker별로 제한된 Tool 준비하기

Researcher에는 웹 검색 Tool을, ChartGenerator에는 전달받은 숫자만 그리는 제한된 차트 Tool을 제공한다. Worker마다 Tool 목록을 분리하면 기능과 실행 권한의 경계가 선명해진다.

- 관련 객체/API: `TavilySearch`, `tool`, `matplotlib.pyplot`
- 입력: 검색 질의 또는 길이가 같은 연도·수치 목록과 제목이다.
- 변환: 검색 결과 수를 제한하거나 전달된 숫자를 선 그래프로 바꾼다.
- 출력: 검색 결과 또는 차트 생성 완료 메시지이다.
- 다음 사용처: 두 Worker의 `tools` 구성이다.

차트 Tool은 외부 코드나 파일 경로를 실행하지 않고, 입력 길이와 자료형처럼 필요한 조건만 확인하도록 구성한다.

In [3]:
import matplotlib.pyplot as plt
from langchain_core.tools import tool
from langchain_tavily import TavilySearch

# LLM 웹 검색 API Tool
tavily_tool = TavilySearch(max_results=3)

# 전달 받은 값을 이용하여 선그래프를 그리는 Tool
@tool
def draw_line_chart(
    years: list[int],
    values: list[float],
    title: str,
) -> str:
    """전달받은 연도와 수치로 영문 제목의 선 그래프를 그린다."""
    _, ax = plt.subplots(figsize=(7, 4))

    ax.plot(years, values, marker="o")
    ax.set_xlabel("Year")
    ax.set_ylabel("Value")
    ax.set_title(title)
    ax.grid(alpha=0.3)
    plt.show()

    return f"Rendered line chart: title={title!r}, years={years}, values={values}"


## MessagesState와 Worker adapter

각 Worker의 내부 Agent State를 상위 LangGraph의 `MessagesState`에 연결하는 adapter를 만든다. 이 adapter가 없으면 Worker의 전체 내부 실행 기록과 상위 graph에 남길 최종 결과의 경계가 불분명해진다.

- 관련 객체/API: `MessagesState`, `agent.invoke`, `HumanMessage`
- 입력: 누적 `messages`, 실행할 Worker, 결과 작성자 이름이다.
- 변환: Worker를 실행하고 마지막 응답 텍스트만 선택한다.
- 출력: 상위 State에 추가할 message 한 건이다.
- 다음 사용처: `StateGraph`의 Researcher와 ChartGenerator Node이다.

In [4]:
import functools

from langchain_core.messages import HumanMessage
from langgraph.graph import MessagesState

# Adapter
def run_agent_node(state:MessagesState, agent, name:str) -> dict:
    result = agent.invoke({"messages": state["messages"]})\

    # Worker Agent의 대답
    worker_text = result["messages"][-1].text

    # worker_text를 HumanMessage로 변환해서 State 누적
    return {
        "messages":[HumanMessage(
            name=name, content=worker_text
        )]
    }

## Researcher와 ChartGenerator Worker 만들기

공유 Chat Model을 만든 뒤 역할별 Agent를 구성한다. Researcher는 출처가 있는 수치를 찾고, ChartGenerator는 State에 전달된 숫자만 시각화하도록 책임을 나눈다.

- 관련 객체/API: `ChatOpenAI`, `create_agent`
- 입력: 공유 model, Worker별 Tool 목록, 역할별 system prompt이다.
- 출력: `research_agent`, `chart_agent`이다.
- 다음 사용처: adapter와 결합해 graph Node로 등록한다.

Researcher에게 차트 Tool을, ChartGenerator에게 검색 Tool을 주지 않는 권한 분리를 유지한다.

## 고정 순서의 Sequential 그래프

Sequential은 요청 내용과 관계없이 `START → Researcher → ChartGenerator → END` 순서를 실행한다. 조사 결과가 먼저 State에 추가되므로 ChartGenerator가 앞 Worker의 수치와 출처를 읽을 수 있다.

- 관련 객체/API: `StateGraph`, `add_node`, `add_edge`, `compile`
- 입력: `MessagesState`와 adapter가 연결된 두 Worker이다.
- 출력: 실행 가능한 `sequential_graph`이다.
- 다음 사용처: GDP 조사와 선 그래프 생성 요청을 실행한다.

각 Node 이름과 실제 실행 함수가 올바르게 대응하는지 확인한다.

## GDP 조사 결과를 선 그래프로 전달하기

한 사용자 요청에 조사 조건과 차트 조건을 함께 넣고 Sequential graph를 실행한다. Node별 변경분을 보면 Researcher의 결과가 ChartGenerator의 입력 근거로 이어지는지 확인할 수 있다.

- 관련 객체/API: `HumanMessage`, `sequential_graph.stream`, `display`, `Markdown`
- 입력: 국가, 연도 범위, 단위, 출처, 차트 형식을 포함한 사용자 message이다.
- 출력: Node 이름과 해당 Node가 추가한 부분 State이다.
- 다음 사용처: 두 Worker 사이 데이터 전달과 고정 실행 순서를 해석한다.

스트림에서는 전체 State가 아니라 각 Node의 최신 update를 읽는다.

## Router의 키워드 분류 규칙

외부 모델을 호출하지 않고 요청 문자열의 키워드로 한 Worker를 선택하는 순수 분류 함수를 만든다. 시각화 표현이 있으면 ChartGenerator를, 그렇지 않으면 Researcher를 선택한다.

- 관련 함수: `select_destination`
- 입력: 사용자 요청 문자열이다.
- 변환: 대소문자를 정규화하고 한글·영문 시각화 키워드 포함 여부를 검사한다.
- 출력: 허용된 Worker 이름 하나이다.
- 다음 사용처: Router Node의 `Command.goto`이다.

간단한 예시 입력으로 두 분기가 모두 선택되는지 확인한다.

## Command로 Router graph 연결하기

Router Node는 마지막 사용자 message를 읽고 선택 결과를 State에 기록하는 동시에 다음 Worker를 지정한다. Worker 실행 후에는 다른 Worker로 이동하지 않고 종료한다.

- 관련 객체/API: `TypedDict`, `Literal`, `Command`, `StateGraph`
- 입력: `messages`가 있는 Router State이다.
- 변환: `select_destination` 결과를 제어 흐름과 관찰용 State 값으로 나눈다.
- 출력: 선택된 Worker로 이동하는 `Command`와 실행 가능한 `router_graph`이다.
- 다음 사용처: 조사 요청과 시각화 요청의 분기 비교이다.

`Command.update`는 데이터 흐름을, `Command.goto`는 제어 흐름을 담당한다.

## 조사 요청과 시각화 요청의 분기 비교

같은 Router graph에 성격이 다른 요청 두 건을 각각 전달한다. 실행마다 새 State를 사용해 앞 요청의 message나 선택 결과가 섞이지 않게 한다.

- 관련 객체/API: `HumanMessage`, `router_graph.invoke`, `display`, `Markdown`
- 입력: 조사 요청 한 건과 숫자가 포함된 시각화 요청 한 건이다.
- 출력: `selected_agent`와 선택된 Worker의 마지막 message이다.
- 확인: 조사 요청은 검색 Worker, 시각화 요청은 차트 Worker 한 곳만 실행되는지 비교한다.

외부 검색과 모델 호출이 발생하는 단계이므로 인증과 네트워크가 준비된 환경에서 실행한다.

## 정리

두 패턴은 같은 Worker를 사용하지만 다음 실행 위치를 정하는 방식이 다르다.

- Sequential은 고정 Edge로 Worker 간 데이터 의존 순서를 보장한다.
- Router는 요청마다 한 Worker만 선택해 불필요한 Tool 호출을 줄인다.
- `MessagesState`는 데이터 흐름이고 Edge와 `Command.goto`는 제어 흐름이다.
- Worker별 `tools` 목록은 기능 목록이자 실행 권한의 경계이다.

조사 후 차트처럼 단계가 정해진 작업에는 Sequential이 적합하다. 조사 또는 시각화 중 하나만 필요한 요청에는 Router가 적합하다.